# The Quantum Fourier Transform and Quantum Phase Estimation with Qiskit

**Qiskit Fall Fest workshop notebook.** Written for Qiskit 2.x, runs in Google Colab, no account needed.

This is the follow-on to *Quantum Computing Essentials with Qiskit*. It assumes you are comfortable with gates (Essentials, Chapter 3), measurement and the primitives (Chapter 4), and above all **interference** (Chapter 7). If the phrase "a phase between two Hadamards becomes a probability" makes sense to you, you are ready.

---

## How to read this notebook

Same rhythm as before. Each chapter follows three steps:

1. We **look** at something in code (a matrix, a circuit, a set of Bloch spheres, a histogram).
2. We **explain** the mathematics behind what we just saw.
3. We **stop and ask you to try** something yourself.

The same conventions are used throughout:

> **Predict.** Before running a cell, write down what you expect to see. Being wrong is the useful part.

> **Physics note.** A short historical or physical aside that connects the code to the real laboratory.

> **Check.** A cell that tests your own answer and tells you whether it is right.

Exercises have hints and full solutions hidden inside collapsible blocks. Open them only after a real attempt.

## The story in one paragraph

The QFT is a change of basis that turns a *number* into a *pattern of phases*, one phase per qubit, like the hands of a clock. Run it backwards and it turns a pattern of phases back into a number you can measure. Phase estimation is the trick that writes the eigenvalue of an unknown unitary into exactly such a pattern of phases. Put the two together and you can read off eigenvalues, which is the engine inside Shor's factoring algorithm and inside quantum chemistry. We finish by factoring 15.

## Contents

| Chapter | Topic |
|---|---|
| 0 | Setting up |
| 1 | From waves to amplitudes: the discrete Fourier transform |
| 2 | The Fourier basis: a binary clock |
| 3 | Building the QFT circuit |
| 4 | The inverse QFT and finding periods |
| 5 | Phase kickback |
| 6 | Quantum phase estimation |
| 7 | Precision: when the phase is not a binary fraction |
| 8 | The library version, energies, and noise |
| 9 | A worked problem: factoring 15 |
| | Where we go next, and further reading |

Each chapter ends with a link to the matching page in the IBM Quantum documentation or the original paper.

## Chapter 0. Setting up

The same three packages as in the Essentials notebook:

- `qiskit`, the core SDK for building circuits, plus visualisation and reference simulators.
- `qiskit-aer`, a fast simulator that can also model noise.
- `qiskit-ibm-runtime`, the client for IBM's cloud hardware, which also ships offline snapshots of real devices.

Run the next cell once. It takes about a minute on a fresh Colab session.

In [ ]:
%pip install -q "qiskit[visualization]" qiskit-aer qiskit-ibm-runtime pylatexenc ipywidgets

In [ ]:
import io
from fractions import Fraction
from math import gcd

import numpy as np
import matplotlib.pyplot as plt

import qiskit
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.circuit.library import QFTGate, PhaseGate, TGate, SGate, UnitaryGate
from qiskit.quantum_info import Statevector, Operator, average_gate_fidelity
from qiskit.visualization import plot_bloch_multivector, plot_histogram
from qiskit.primitives import StatevectorSampler
from ipywidgets import interact, FloatSlider, IntSlider

print("Qiskit version:", qiskit.__version__)

sampler = StatevectorSampler()


# The two helpers from the Essentials notebook.

def fig_to_img(fig):
    # Render a matplotlib Figure to an image array so we can place
    # several Bloch-sphere figures side by side.
    buf = io.BytesIO()
    fig.savefig(buf, format="png", bbox_inches="tight", dpi=100)
    buf.seek(0)
    return plt.imread(buf)


def check(label, test):
    # Run a test (a function returning True/False) and report the result.
    # If your variable is still None or the code raises, we say "Not yet".
    try:
        ok = bool(test())
    except Exception:
        ok = False
    print(("Correct:  " if ok else "Not yet:  ") + label)

## Chapter 1. From waves to amplitudes: the discrete Fourier transform

The classical **discrete Fourier transform** (DFT) takes a list of $N$ complex numbers $x_0,\dots,x_{N-1}$ and rewrites it as a sum of waves. In the normalisation we will use,

$$y_k = \frac{1}{\sqrt N}\sum_{j=0}^{N-1} x_j\, \omega^{jk}, \qquad \omega = e^{2\pi i/N}.$$

The number $\omega$ is an $N$-th root of unity: a point on the unit circle, $1/N$ of the way round. Raising it to the power $jk$ walks around the circle in steps of size $j$.

The **quantum Fourier transform** is the same linear map, applied to the *amplitudes* of an $n$-qubit state, with $N = 2^n$. On a basis state it reads

$$\mathrm{QFT}\,|j\rangle = \frac{1}{\sqrt N}\sum_{k=0}^{N-1} \omega^{jk}\,|k\rangle .$$

As a matrix,

$$F_N = \frac{1}{\sqrt N}\begin{pmatrix}
1 & 1 & 1 & \cdots & 1\\
1 & \omega & \omega^2 & \cdots & \omega^{N-1}\\
1 & \omega^2 & \omega^4 & \cdots & \omega^{2(N-1)}\\
\vdots & & & \ddots & \vdots\\
1 & \omega^{N-1} & \omega^{2(N-1)} & \cdots & \omega^{(N-1)^2}
\end{pmatrix}.$$

Every column has the same length and the columns are orthogonal, so $F_N$ is **unitary**. It is a legal quantum gate.

> **Physics note.** Joseph Fourier introduced these series around 1807 to solve the heat equation, and was widely disbelieved. The *fast* Fourier transform, which computes the DFT in $O(N\log N)$ steps instead of $O(N^2)$, was published by Cooley and Tukey in 1965, though Gauss had used the same trick around 1805 to compute the orbits of asteroids. In 1994 Don Coppersmith showed that the quantum version needs only $O(n^2) = O(\log^2 N)$ gates. That is exponentially fewer than the FFT, but with a catch that we will meet in Chapter 4: the output lives in amplitudes, which you cannot simply read.

We start by writing $F_N$ down directly in NumPy.

In [ ]:
def dft_matrix(N):
    # The N x N quantum Fourier transform matrix, built straight from the definition.
    j, k = np.meshgrid(np.arange(N), np.arange(N))
    return np.exp(2j * np.pi * j * k / N) / np.sqrt(N)

F4 = dft_matrix(4)
print(np.round(F4 * 2, 3))          # times sqrt(4) = 2 to make the entries easy to read
print()
print("Unitary?", np.allclose(F4.conj().T @ F4, np.eye(4)))

For one qubit, $N=2$ and $\omega = e^{i\pi} = -1$, so

$$F_2 = \frac{1}{\sqrt2}\begin{pmatrix}1&1\\1&-1\end{pmatrix} = H.$$

The Hadamard gate *is* the one-qubit Fourier transform. Everything we said about H in the Essentials notebook was secretly a statement about Fourier transforms.

In [ ]:
from qiskit.circuit.library import HGate
print("F_2 equals H?", np.allclose(dft_matrix(2), Operator(HGate()).data))

Qiskit has the QFT built in as `QFTGate`. Its matrix should equal ours exactly.

One warning about conventions before we compare with NumPy. NumPy's `np.fft.fft` uses $e^{-2\pi i jk/N}$ and no normalisation, which is the opposite sign to the quantum convention. The quantum QFT therefore matches NumPy's **inverse** FFT, multiplied by $\sqrt N$. Different books choose differently; the physics is the same.

In [ ]:
n = 3
N = 2**n

F_qiskit = Operator(QFTGate(n)).data
print("QFTGate matches our matrix?          ", np.allclose(F_qiskit, dft_matrix(N)))

v = np.random.default_rng(1).normal(size=N) + 1j * np.random.default_rng(2).normal(size=N)
print("QFT v equals sqrt(N) * numpy ifft(v)?", np.allclose(F_qiskit @ v, np.sqrt(N) * np.fft.ifft(v)))

### Seeing a column

The amplitudes of $\mathrm{QFT}|j\rangle$ are $\omega^{jk}/\sqrt N$ for $k=0,1,\dots,N-1$. Plotted in the complex plane, they march around the unit circle, $j$ steps of $1/N$ of a turn at a time. Larger $j$ means faster rotation: $j$ is a **frequency**.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
N = 8
for ax, j in zip(axes, [1, 2, 3]):
    amps = dft_matrix(N)[:, j] * np.sqrt(N)          # column j, rescaled to the unit circle
    ax.add_patch(plt.Circle((0, 0), 1, fill=False, ls="--", color="gray"))
    labels = {}                                       # arrows that land on the same point share a label
    for k, a in enumerate(amps):
        ax.annotate("", xy=(a.real, a.imag), xytext=(0, 0),
                    arrowprops=dict(arrowstyle="->", color=plt.cm.viridis(k / N)))
        labels.setdefault((round(a.real, 6), round(a.imag, 6)), []).append(str(k))
    for (re, im), ks in labels.items():
        ax.text(1.22 * re, 1.22 * im, ",".join(ks), ha="center", va="center")
    ax.set_xlim(-1.4, 1.4); ax.set_ylim(-1.4, 1.4); ax.set_aspect("equal"); ax.axis("off")
    ax.set_title(f"amplitudes of QFT|{j}>, N = {N}")
plt.show()

For $j=1$ each label $k$ is one-eighth of a turn further round. For $j=2$ the arrows go round twice and land on top of each other in pairs. For $j=3$ they go round three times.

> **Predict.** What is $\mathrm{QFT}|0\rangle$? Look at the first column of $F_N$ before you run the next cell.

In [ ]:
qc = QuantumCircuit(3)
qc.append(QFTGate(3), range(3))
display(Statevector(qc).draw("latex"))

Frequency zero means "no rotation at all": every amplitude is $1/\sqrt8$. So $\mathrm{QFT}|0\ldots0\rangle = H^{\otimes n}|0\ldots0\rangle$, the uniform superposition. For this one input the QFT and a layer of Hadamards agree. For every other input they do not.

### Exercise 1: the four-point transform by hand

**Goal.** Write the $4\times4$ matrix $F_4$ by hand as a NumPy array `ex1_F4`. Do not call `dft_matrix`. For $N=4$ the root of unity is $\omega = e^{2\pi i/4}$, a quarter-turn.

<details>
<summary>Hint</summary>

A quarter-turn on the unit circle is $i$. So the entries are powers of $i$: $1, i, -1, -i$, repeating. Entry $(k, j)$ is $i^{jk}/2$. Row 0 and column 0 are all ones (before the $1/2$).
</details>

In [ ]:
ex1_F4 = None      # np.array([[...], [...], [...], [...]]) / 2

In [ ]:
check("ex1_F4 equals the four-point QFT matrix", lambda: np.allclose(ex1_F4, dft_matrix(4)))

<details>
<summary>Solution</summary>

```python
ex1_F4 = np.array([[1,  1,  1,  1],
                   [1,  1j, -1, -1j],
                   [1, -1,  1, -1],
                   [1, -1j, -1,  1j]]) / 2
```

Row $k$, column $j$ holds $i^{jk}/2$. The matrix is symmetric because $jk = kj$. Notice row 2: $1,-1,1,-1$. That is the fastest wave that four samples can represent (the Nyquist frequency).
</details>

Further reading: the [QFTGate entry in the circuit library](https://quantum.cloud.ibm.com/docs/en/api/qiskit/circuit_library) and the IBM Quantum Learning course [Fundamentals of quantum algorithms](https://quantum.cloud.ibm.com/learning/en/courses/fundamentals-of-quantum-algorithms).

## Chapter 2. The Fourier basis: a binary clock

Apply the QFT to every computational basis state $|x\rangle$ and you get a new orthonormal basis, the **Fourier basis**. We write its states with a tilde:

$$|\tilde x\rangle = \mathrm{QFT}\,|x\rangle = \frac{1}{\sqrt N}\sum_{y=0}^{N-1} e^{2\pi i\,xy/N}\,|y\rangle .$$

These states have a remarkable property. **Every one of them is a product state.** No entanglement at all. Here is why.

Write the index $y$ in binary, using Qiskit's little-endian convention (qubit 0 is the least significant bit):

$$y = \sum_{k=0}^{n-1} y_k\,2^k, \qquad |y\rangle = |y_{n-1}\rangle\otimes\cdots\otimes|y_1\rangle\otimes|y_0\rangle .$$

Then the phase splits into a product, one factor per qubit,

$$e^{2\pi i\,xy/N} = \prod_{k=0}^{n-1} e^{2\pi i\,x\,y_k\,2^k/N},$$

and the sum over all $y$ becomes a sum over each bit separately. The result is

$$\boxed{\;|\tilde x\rangle = \bigotimes_{k=n-1}^{0}\ \frac{|0\rangle + e^{i\varphi_k}|1\rangle}{\sqrt2}, \qquad \varphi_k = \frac{2\pi\,x\,2^k}{N}.\;}$$

Each qubit sits on the **equator** of its Bloch sphere, rotated by an angle $\varphi_k$. Qubit 0 rotates by $2\pi x/N$, qubit 1 by twice that, qubit 2 by four times that, and so on. The number $x$ is stored not in which qubits are 0 or 1, but in **how far each qubit has turned**.

> **Physics note.** Think of the hands of a clock. As $x$ increases by one, qubit 0 advances by $1/N$ of a turn (the hour hand), and qubit $n-1$ advances by half a turn (a very fast second hand). In physics the computational basis and the Fourier basis play the roles of **position** and **momentum**: a state that is sharp in one is completely spread out in the other. That is the discrete version of Heisenberg's uncertainty relation, and it is why $\mathrm{QFT}|0\rangle$ was the uniform superposition.

Let us check the formula numerically before believing it.

In [ ]:
def fourier_state_from_formula(x, n):
    # Build |x~> as a tensor product of single-qubit equator states.
    N = 2**n
    state = np.array([1.0 + 0j])
    for k in range(n):                               # qubit 0 first ...
        phi_k = 2 * np.pi * x * 2**k / N
        qubit = np.array([1, np.exp(1j * phi_k)]) / np.sqrt(2)
        state = np.kron(qubit, state)                # ... ends up as the rightmost factor
    return state

n = 3
all_ok = all(np.allclose(fourier_state_from_formula(x, n), dft_matrix(2**n)[:, x]) for x in range(2**n))
print("Product formula correct for all 8 basis states?", all_ok)

### Watching the clock

Below are the Fourier states $|\tilde1\rangle$, $|\tilde2\rangle$ and $|\tilde3\rangle$ on three qubits. Qubit 0 is on the **left** in each row.

> **Predict.** For $|\tilde1\rangle$: qubit 0 turns by $2\pi/8$, qubit 1 by $2\pi/4$, qubit 2 by $\pi$. Where on each equator should the three arrows point?

In [ ]:
for x in [1, 2, 3]:
    qc = QuantumCircuit(3)
    qc.prepare_state(Statevector.from_int(x, 8))     # start in |x>
    qc.append(QFTGate(3), range(3))
    display(plot_bloch_multivector(Statevector(qc), title=f"|{x}~>  =  QFT|{x}>"))

### Interactive: turn the clock

Drag the slider through $x = 0, 1, \dots, 15$ for a four-qubit register. Watch qubit 3 (the fast hand) flip between $+x$ and $-x$ at every step, while qubit 0 (the slow hand) creeps round by one-sixteenth of a turn. Only after sixteen steps are all hands back at the start.

In [ ]:
@interact(x=IntSlider(min=0, max=15, step=1, value=1, description="x"))
def fourier_clock(x):
    n = 4
    sv = Statevector(fourier_state_from_formula(x, n))
    print("phases phi_k / 2pi  (qubit 0 .. 3):",
          [f"{(x * 2**k / 2**n) % 1:.4f}" for k in range(n)])
    fig = plot_bloch_multivector(sv, title=f"|{x}~> on four qubits")
    display(fig)
    plt.close(fig)

### Binary fractions

The phase on each qubit has a neat form in binary. Write $x = x_{n-1}\dots x_1x_0$ in bits, and use the notation $0.b_1b_2b_3\ldots = b_1/2 + b_2/4 + b_3/8 + \dots$ for a binary fraction. Because whole turns do nothing, only the fractional part of $x\,2^k/N$ matters, and

$$\frac{\varphi_{n-1}}{2\pi} = 0.x_0,\qquad \frac{\varphi_{n-2}}{2\pi} = 0.x_1x_0,\qquad \dots,\qquad \frac{\varphi_0}{2\pi} = 0.x_{n-1}\dots x_1x_0 .$$

So the fast hand (qubit $n-1$) depends only on the last bit of $x$, and the slow hand (qubit 0) depends on every bit. Keep this in mind. It tells us exactly what the circuit in Chapter 3 has to do.

### Exercise 2: building a Fourier state without the QFT

Because $|\tilde x\rangle$ is a product state, you do not need any two-qubit gates to *prepare* it.

**Goal.** Build a three-qubit circuit `ex2_circuit` using only `h` and `p` gates that prepares $|\tilde5\rangle$.

<details>
<summary>Hint</summary>

`h` puts a qubit on the equator at $+x$. `p(phi, k)` then turns qubit $k$ by $\varphi$ about z. Use $\varphi_k = 2\pi\cdot5\cdot2^k/8$.
</details>

In [ ]:
ex2_circuit = None

In [ ]:
def _ex2():
    target = QuantumCircuit(3)
    target.x([0, 2])                       # |5> = |101>
    target.append(QFTGate(3), range(3))
    ops = set(ex2_circuit.count_ops())
    return Statevector(ex2_circuit).equiv(Statevector(target)) and ops <= {"h", "p"}

check("ex2_circuit prepares |5~> using only h and p", _ex2)

<details>
<summary>Solution</summary>

```python
ex2_circuit = QuantumCircuit(3)
for k in range(3):
    ex2_circuit.h(k)
    ex2_circuit.p(2 * np.pi * 5 * 2**k / 8, k)
```

The three phases are $5\pi/4$, $5\pi/2 \equiv \pi/2$ and $5\pi \equiv \pi$. In binary fractions: $0.101$, $0.01$ and $0.1$, exactly the pattern from the table above with $x = 101$.
</details>

**Stretch.** If preparing a Fourier state is this easy, why does the QFT circuit need two-qubit gates at all? (Answer: here *you* knew $x$ and computed the angles classically. The QFT has to compute them from qubits whose values it does not know, possibly in superposition.)

Further reading: chapter 5 of Nielsen and Chuang derives the product formula in big-endian convention. Translating it into Qiskit's ordering is a good exercise.

## Chapter 3. Building the QFT circuit

We now need a circuit that, given an unknown $|x\rangle$, puts phase $2\pi\cdot 0.x_j\ldots x_0$ on the right qubit. Two ingredients are enough.

**Ingredient 1: the Hadamard writes one bit as a phase.** On a basis state,

$$H|x_j\rangle = \frac{|0\rangle + e^{i\pi x_j}|1\rangle}{\sqrt2} = \frac{|0\rangle + e^{2\pi i\,(0.x_j)}|1\rangle}{\sqrt2}.$$

**Ingredient 2: the controlled-phase gate adds more bits.** The gate $CP(\lambda)$ is

$$CP(\lambda) = \begin{pmatrix}1&0&0&0\\0&1&0&0\\0&0&1&0\\0&0&0&e^{i\lambda}\end{pmatrix}.$$

It multiplies by $e^{i\lambda}$ only when *both* qubits are 1. If qubit $m$ holds a definite bit $x_m$, a $CP(\pi/2^{d})$ between qubit $m$ and a qubit on the equator turns that qubit by an extra $2\pi\cdot x_m/2^{d+1}$. That appends the bit $x_m$, $d$ places further along the binary fraction.

**The recipe.** Work from the top qubit down. For $j = n-1, n-2, \dots, 0$:

1. Apply $H$ to qubit $j$. It now carries phase $2\pi\cdot0.x_j$.
2. For each lower qubit $m<j$, apply $CP\big(\pi/2^{\,j-m}\big)$ between $m$ and $j$. Now qubit $j$ carries $2\pi\cdot0.x_jx_{j-1}\ldots x_0$.

The lower qubits are still untouched basis states at that point, which is why we go from the top down. When we are done, qubit $j$ holds the phase that the product formula wanted on qubit $n-1-j$. The order is reversed, so the last step is:

3. Swap qubit $k$ with qubit $n-1-k$.

> **Physics note.** The controlled-phase gate is **symmetric**: its matrix only cares whether both qubits are 1, so "control" and "target" are interchangeable. Physically, on many superconducting devices the native two-qubit interaction is a conditional phase of exactly this kind, produced by bringing two transmons briefly into resonance. When you draw $CP$ in Qiskit you see two dots and no target symbol, for this reason.

We build it.

In [ ]:
def qft_manual(n, swaps=True, barriers=True):
    qc = QuantumCircuit(n, name="QFT")
    for j in reversed(range(n)):                 # top qubit first
        qc.h(j)
        for m in reversed(range(j)):             # every lower qubit adds one more bit
            qc.cp(np.pi / 2**(j - m), m, j)
        if barriers:
            qc.barrier()                         # only for readability of the drawing
    if swaps:
        for k in range(n // 2):
            qc.swap(k, n - 1 - k)
    return qc

qft4 = qft_manual(4)
qft4.draw("mpl", fold=-1)

Read the drawing column by column. Qubit 3 gets an $H$ and then three controlled phases of $\pi/2$, $\pi/4$, $\pi/8$ from qubits 2, 1, 0. Qubit 2 gets an $H$ and two phases. And so on, down to qubit 0, which gets only an $H$. Then the swaps.

Now the test that matters.

In [ ]:
for n in range(1, 6):
    ok_formula = np.allclose(Operator(qft_manual(n)).data, dft_matrix(2**n))
    ok_qiskit  = Operator(qft_manual(n)).equiv(Operator(QFTGate(n)))
    print(f"n = {n}:  matches F_N? {ok_formula}   matches QFTGate? {ok_qiskit}")

### Symmetry of the controlled phase

> **Predict.** Is `cp(lam, 0, 1)` the same operator as `cp(lam, 1, 0)`?

In [ ]:
a = QuantumCircuit(2); a.cp(0.7, 0, 1)
b = QuantumCircuit(2); b.cp(0.7, 1, 0)
print("Same operator?", Operator(a).equiv(Operator(b)))

### Counting the cost

The circuit has $n$ Hadamards, $n(n-1)/2$ controlled phases and $\lfloor n/2\rfloor$ swaps: $O(n^2)$ gates in total. The classical FFT on a vector of length $N = 2^n$ needs about $N\log_2 N = n\,2^n$ operations.

In [ ]:
print(" n | QFT gates | FFT operations (n 2^n)")
print("---+-----------+-----------------------")
for n in [2, 4, 8, 16, 32, 64]:
    ops = qft_manual(n).count_ops()
    gates = ops["h"] + ops.get("cp", 0) + ops.get("swap", 0)
    print(f"{n:2d} | {gates:9d} | {n * 2**n:.3g}")

At $n=64$ the QFT needs about two thousand gates. The FFT on the same amount of data would need around $10^{21}$ operations. Hold on to that number, and to the caveat from Chapter 1: the QFT does not *give* you the $2^n$ output amplitudes. Chapter 4 shows what it does give you.

### The approximate QFT

Look at the angles: $\pi/2, \pi/4, \pi/8, \dots, \pi/2^{n-1}$. For large $n$ the last ones are tiny rotations, far smaller than the error of any real gate. Coppersmith noticed in 1994 that you can simply **drop** every controlled phase with $j - m > d$ for some cut-off $d$, and the result is still an excellent approximation. The gate count falls from $O(n^2)$ to $O(n\,d)$, and $d \approx \log_2 n$ is enough in practice.

### Exercise 3: an approximate QFT

**Goal.** Write `aqft(n, d)` that is the same as `qft_manual` except it only applies $CP(\pi/2^{\,j-m})$ when $j - m \le d$. Then build `ex3_circuit = aqft(6, 2)`.

The check compares your circuit with the exact QFT using the **average gate fidelity**, a number between 0 and 1 that measures how close two unitaries are (1 means identical up to global phase).

<details>
<summary>Hint</summary>

Copy `qft_manual` and add one `if` inside the inner loop. For $n=6$, $d=2$ you should end up with 9 controlled phases instead of 15.
</details>

In [ ]:
def aqft(n, d):
    return None

ex3_circuit = None

In [ ]:
check("ex3_circuit has exactly 9 controlled phases",
      lambda: ex3_circuit.count_ops()["cp"] == 9)
check("its average gate fidelity with the exact QFT is above 0.85",
      lambda: average_gate_fidelity(Operator(ex3_circuit), Operator(QFTGate(6))) > 0.85)

<details>
<summary>Solution</summary>

```python
def aqft(n, d):
    qc = QuantumCircuit(n, name="AQFT")
    for j in reversed(range(n)):
        qc.h(j)
        for m in reversed(range(j)):
            if j - m <= d:                         # keep only the larger rotations
                qc.cp(np.pi / 2**(j - m), m, j)
    for k in range(n // 2):
        qc.swap(k, n - 1 - k)
    return qc

ex3_circuit = aqft(6, 2)
print(average_gate_fidelity(Operator(ex3_circuit), Operator(QFTGate(6))))
```

The fidelity is about 0.86 with only 9 of the 15 controlled phases. With $d = 3$ it is already about 0.98, and with $d = 4$ above 0.998.
</details>

**Stretch.** Transpile `qft_manual(6)` and `aqft(6, 2)` for `FakeSherbrooke` (see Essentials, Chapter 8) and compare the number of two-qubit gates. On a noisy device, which do you expect to give the *more* accurate answer?

Further reading: D. Coppersmith, [An approximate Fourier transform useful in quantum factoring](https://arxiv.org/abs/quant-ph/0201067) (1994), and the [Qiskit synthesis module](https://quantum.cloud.ibm.com/docs/en/api/qiskit/synthesis), whose `synth_qft_full` has an `approximation_degree` argument that does exactly this.

## Chapter 4. The inverse QFT and finding periods

### Reading a phase pattern

The QFT is unitary, so it has an inverse $\mathrm{QFT}^\dagger$. In Qiskit, `qc.inverse()` reverses the gate order and conjugates every angle. Its job is the one we care about most:

$$\mathrm{QFT}^\dagger\,|\tilde x\rangle = |x\rangle .$$

If someone hands us $n$ qubits sitting on their equators with the clock-hand pattern of Chapter 2, the inverse QFT turns that pattern into the plain number $x$, which a measurement then reads perfectly. **Writing a number as phases and reading it back with the inverse QFT is the whole idea of phase estimation.** Here is a first demonstration, using the product state from Exercise 2.

In [ ]:
iqft3 = qft_manual(3).inverse()
iqft3.name = "QFT†"

qc = QuantumCircuit(3)
for k in range(3):                                   # write x = 5 into the phases (Exercise 2)
    qc.h(k)
    qc.p(2 * np.pi * 5 * 2**k / 8, k)
qc.barrier()
qc.compose(iqft3, inplace=True)                      # read it back
qc.measure_all()
display(qc.draw("mpl", fold=-1))

counts = sampler.run([qc], shots=1000).result()[0].data.meas.get_counts()
print(counts, "->  x =", int(max(counts, key=counts.get), 2))

Every shot returns `101`, which is 5. The inverse QFT is an interferometer: the phases on the eight branches of the state line up so that all the wrong answers cancel and the right one reinforces, as in Essentials Chapter 7 but on $2^n$ paths at once.

### Periodic states

Now the most important property of the Fourier transform: it turns **periodicity** into **sharp peaks**. Take a state that is an equal superposition of every $r$-th basis state, starting from an offset $s$:

$$|\psi_{r,s}\rangle = \sqrt{\frac{r}{N}}\sum_{m=0}^{N/r-1} |s + m r\rangle .$$

If $r$ divides $N$, the QFT of this state is supported only on the multiples of $N/r$:

$$\mathrm{QFT}\,|\psi_{r,s}\rangle = \frac{1}{\sqrt r}\sum_{\ell=0}^{r-1} e^{2\pi i\, s\ell/r}\,\Big|\ell\,\tfrac{N}{r}\Big\rangle .$$

Two things to notice. The peaks are at spacing $N/r$, so they reveal the period. And the offset $s$ only appears in the phases, so it **does not change the measurement probabilities** at all.

> **Physics note.** This is diffraction. Light passing through a grating with slit spacing $d$ produces bright spots at angles set by $1/d$: a periodic object becomes a set of sharp peaks in the Fourier domain. Shifting the whole grating sideways does not move the spots. In 1913 William Henry and Lawrence Bragg used exactly this to read off the spacing of atoms in a crystal from X-ray spots, and X-ray crystallography later revealed the structure of DNA. Shor's algorithm uses the same physics to read off the period of a function.

In [ ]:
def periodic_state(n, r, s):
    N = 2**n
    vec = np.zeros(N)
    vec[s::r] = 1
    return Statevector(vec / np.linalg.norm(vec))

n, r = 4, 4
fig, axes = plt.subplots(2, 2, figsize=(12, 6))
for col, s in enumerate([0, 3]):
    before = periodic_state(n, r, s)
    after  = before.evolve(QFTGate(n))
    axes[0, col].bar(range(2**n), before.probabilities()); axes[0, col].set_title(f"before QFT: period {r}, offset {s}")
    axes[1, col].bar(range(2**n), after.probabilities(), color="C1"); axes[1, col].set_title("after QFT")
    for ax in axes[:, col]:
        ax.set_xticks(range(2**n)); ax.set_ylim(0, 1)
plt.tight_layout(); plt.show()

Different offsets, identical peaks at 0, 4, 8, 12, which are multiples of $16/4 = 4$. If you measured this state you would get one of those four numbers at random, and from a few samples you could recover the period.

That is the honest answer to the caveat from Chapter 1. The QFT does not hand you all $N$ Fourier coefficients. It hands you a **sample** from the Fourier spectrum. When the spectrum has a few sharp peaks, a few samples are enough.

### Exercise 4: predicting the peaks

**Goal.** On four qubits ($N = 16$), a state has period $r = 2$. Without running anything, write the list `ex4_peaks` of all outcomes that can appear after the QFT, in increasing order.

<details>
<summary>Hint</summary>

Peaks sit at multiples of $N/r$, and there are exactly $r$ of them.
</details>

In [ ]:
ex4_peaks = None       # e.g. [0, ...]

In [ ]:
def _ex4():
    probs = periodic_state(4, 2, 1).evolve(QFTGate(4)).probabilities()
    return list(ex4_peaks) == [i for i, p in enumerate(probs) if p > 1e-9]

check("you found every possible outcome and nothing else", _ex4)

<details>
<summary>Solution</summary>

```python
ex4_peaks = [0, 8]
```

$N/r = 8$, so the peaks are 0 and 8. A period-2 pattern is the fastest one possible, and its spectrum has a component at the highest frequency, the Nyquist frequency $N/2$, just like row 2 of $F_4$ in Exercise 1.
</details>

**Stretch.** Try $r = 3$ with `periodic_state(4, 3, 0)`. Three does not divide sixteen, so the peaks can no longer sit exactly on integers. What happens to them? Chapter 7 explains the shape you see.

Further reading: the lesson on phase estimation and factoring in [Fundamentals of quantum algorithms](https://quantum.cloud.ibm.com/learning/en/courses/fundamentals-of-quantum-algorithms).

## Chapter 5. Phase kickback

We now switch from the QFT to the question it will help us answer.

**The problem.** We are given a unitary $U$ and one of its eigenstates $|\psi\rangle$. Every eigenvalue of a unitary has modulus 1, so we can write

$$U|\psi\rangle = e^{2\pi i\theta}|\psi\rangle, \qquad 0 \le \theta < 1 .$$

We want $\theta$. The difficulty is that $e^{2\pi i\theta}$ multiplies the *whole* state. It is a global phase, and the Essentials notebook taught us that a global phase is invisible to every measurement.

**The trick.** Make the phase relative by applying $U$ only *conditionally*. Put a control qubit in $|+\rangle$ and apply controlled-$U$:

$$\frac{|0\rangle + |1\rangle}{\sqrt2}\,|\psi\rangle \ \xrightarrow{\ C\text{-}U\ }\ \frac{|0\rangle|\psi\rangle + |1\rangle\,U|\psi\rangle}{\sqrt2} = \frac{|0\rangle + e^{2\pi i\theta}|1\rangle}{\sqrt2}\otimes|\psi\rangle .$$

The target is unchanged. The phase has moved onto the **control** qubit, where it is now a relative phase between $|0\rangle$ and $|1\rangle$. This is **phase kickback**.

> **Physics note.** The idea that a "control" qubit can be changed by the gate it controls surprises everyone the first time. It is already hidden in a fact from Essentials Chapter 3: in the X basis, a CNOT's control and target swap roles. Kickback was used implicitly in Deutsch-Jozsa and in Kitaev's 1995 phase estimation, and was named and made into the unifying principle of quantum algorithms by Cleve, Ekert, Macchiavello and Mosca in 1998. Their paper is short and still the clearest explanation of why QFT-based algorithms work.

We use $U = T$, whose eigenstate $|1\rangle$ has $T|1\rangle = e^{i\pi/4}|1\rangle$, so $\theta = 1/8$.

In [ ]:
def kickback_demo(gate, target_prep):
    qc = QuantumCircuit(2)                    # qubit 0 = control, qubit 1 = target
    qc.compose(target_prep, [1], inplace=True)
    qc.h(0)
    before = Statevector(qc)
    qc.append(gate.control(1), [0, 1])
    after = Statevector(qc)
    display(qc.draw("mpl"))
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    for ax, (title, sv) in zip(axes, [("BEFORE controlled-U", before), ("AFTER controlled-U", after)]):
        f = plot_bloch_multivector(sv, title=title)
        ax.imshow(fig_to_img(f)); ax.axis("off"); plt.close(f)
    plt.show()

prep_one = QuantumCircuit(1); prep_one.x(0)          # eigenstate |1> of T
kickback_demo(TGate(), prep_one)

The control (qubit 0, left) has turned by $\pi/4$ about z, exactly $2\pi\theta$. The target (qubit 1, right) has not moved.

> **Predict.** Repeat with the target in $|0\rangle$, which is also an eigenstate of T. What is its eigenvalue, and what happens to the control?

In [ ]:
kickback_demo(TGate(), QuantumCircuit(1))

Eigenvalue 1, so $\theta = 0$ and nothing is kicked back. The control stays at $+x$.

### Kickback needs an eigenstate

What if the target is **not** an eigenstate? Try $|+\rangle$, which is an equal mix of the two eigenstates of T with different eigenvalues.

In [ ]:
prep_plus = QuantumCircuit(1); prep_plus.h(0)
kickback_demo(TGate(), prep_plus)

Both arrows have shrunk inside their spheres. The control has become **entangled** with the target, because the two eigen-components kicked back different phases. Kickback gives a clean phase only when the target is an eigenstate. (In Chapter 8 we will see that this "failure" is actually useful: it turns phase estimation into a measurement of which eigenstate you are in.)

### Turning the phase into a probability: the Hadamard test

A relative phase is still invisible to a Z measurement, so we finish with a second Hadamard on the control. This is exactly the interferometer from Essentials Chapter 7 with $\varphi = 2\pi\theta$:

$$P(0) = \cos^2(\pi\theta).$$

In [ ]:
thetas = np.linspace(0, 1, 41)
p0 = []
for th in thetas:
    qc = QuantumCircuit(2, 1)
    qc.x(1)                                   # eigenstate |1>
    qc.h(0)
    qc.cp(2 * np.pi * th, 0, 1)               # controlled-P(2 pi theta): eigenvalue e^{2 pi i theta} on |1>
    qc.h(0)
    qc.measure(0, 0)
    c = sampler.run([qc], shots=2000).result()[0].data.c.get_counts()
    p0.append(c.get("0", 0) / 2000)

fine = np.linspace(0, 1, 300)
plt.figure(figsize=(7, 3.5))
plt.plot(thetas, p0, "o", label="simulated shots")
plt.plot(fine, np.cos(np.pi * fine)**2, label=r"theory $\cos^2(\pi\theta)$")
plt.xlabel(r"eigenphase $\theta$"); plt.ylabel("P(0) on the control")
plt.title("The Hadamard test"); plt.legend(); plt.grid(alpha=.3); plt.show()

One control qubit gives us one number, $\cos^2(\pi\theta)$, and only statistically. It cannot distinguish $\theta$ from $1-\theta$, and to pin $\theta$ down to $m$ digits we would need on the order of $10^{2m}$ shots. Chapter 6 does exponentially better.

### Exercise 5: what gets kicked back?

**Goal.** Take $U = S$ and the eigenstate $|1\rangle$. Set `ex5_theta` to the value of $\theta$ (in turns, so that the eigenvalue is $e^{2\pi i\theta}$). Then build the two-qubit kickback circuit `ex5_circuit` (control = qubit 0, target = qubit 1, no measurements, no final Hadamard) and the check will compare the control's Bloch vector with your prediction.

<details>
<summary>Hint</summary>

$S = \mathrm{diag}(1, i)$ and $i = e^{i\pi/2}$. What fraction of a full turn is $\pi/2$? For the circuit, use `SGate().control(1)` or `qc.cs(0, 1)`.
</details>

In [ ]:
ex5_theta   = None
ex5_circuit = None

In [ ]:
def _ex5():
    from qiskit.quantum_info import partial_trace, DensityMatrix
    rho = partial_trace(Statevector(ex5_circuit), [1])                     # state of the control
    expected = DensityMatrix(np.array([1, np.exp(2j * np.pi * ex5_theta)]) / np.sqrt(2))
    return np.isclose(ex5_theta, 0.25) and np.allclose(rho.data, expected.data)

check("theta = 1/4 and the control carries exactly that phase", _ex5)

<details>
<summary>Solution</summary>

```python
ex5_theta = 0.25

ex5_circuit = QuantumCircuit(2)
ex5_circuit.x(1)            # eigenstate |1>
ex5_circuit.h(0)            # control in |+>
ex5_circuit.cs(0, 1)        # controlled-S
```

The control ends at the $+y$ point of its sphere, the state `"r"` from the Essentials notebook. A Z measurement now gives 50/50; a Hadamard test gives $\cos^2(\pi/4) = 1/2$ as well. Neither alone tells $\theta = 1/4$ apart from $\theta = 3/4$. A Y measurement would. Phase estimation will tell them apart automatically.
</details>

Further reading: R. Cleve, A. Ekert, C. Macchiavello and M. Mosca, [Quantum algorithms revisited](https://arxiv.org/abs/quant-ph/9708016) (1998).

## Chapter 6. Quantum phase estimation

Here is the key observation. Instead of one control qubit, use $t$ of them, the **counting register**. Let counting qubit $k$ control not $U$ but $U^{2^k}$. Since $U^{2^k}|\psi\rangle = e^{2\pi i\,\theta\,2^k}|\psi\rangle$, kickback puts the phase

$$\varphi_k = 2\pi\,\theta\,2^k$$

on counting qubit $k$. Now compare with the Fourier state from Chapter 2, whose qubit $k$ carries $\varphi_k = 2\pi\,x\,2^k/2^t$. They are **the same pattern** if

$$x = \theta\,2^t .$$

So after the controlled powers, the counting register is exactly the Fourier state $|\widetilde{\theta 2^t}\rangle$. The inverse QFT from Chapter 4 turns it into the basis state $|\theta\,2^t\rangle$, and a measurement reads it. In one line:

$$|0\rangle^{\otimes t}|\psi\rangle \xrightarrow{H^{\otimes t}} \xrightarrow{\ \prod_k C\text{-}U^{2^k}\ } |\widetilde{\theta 2^t}\rangle|\psi\rangle \xrightarrow{\ \mathrm{QFT}^\dagger\ } |\theta 2^t\rangle|\psi\rangle .$$

The estimate is $\tilde\theta = x/2^t$, where $x$ is the measured integer. Each extra counting qubit gives one more binary digit of $\theta$.

> **Physics note.** Qubit $k$ "sees" $U$ applied $2^k$ times, so its phase accumulates $2^k$ times faster. In metrology this is the **Heisenberg limit**: with total resources $T$ (here, the total number of applications of $U$) the error falls like $1/T$, compared with $1/\sqrt T$ for repeating independent experiments as in the Hadamard test. Atomic clocks and gravitational-wave detectors chase exactly this scaling. Alexei Kitaev formulated phase estimation in 1995; the circuit below, with the inverse QFT at the end, is the form popularised by Cleve and co-authors.

Now we build a general QPE circuit. The arguments are the gate $U$, a small circuit that prepares the eigenstate, and the number of counting qubits $t$.

In [ ]:
def qpe_circuit(U, eigenstate_prep, t, measure=True, inverse_qft=True):
    m = U.num_qubits
    count  = QuantumRegister(t, "count")
    target = QuantumRegister(m, "target")
    qc = QuantumCircuit(count, target)

    qc.compose(eigenstate_prep, target, inplace=True)   # prepare |psi>
    qc.h(count)                                          # every counting qubit in |+>
    qc.barrier()

    for k in range(t):                                   # counting qubit k controls U^(2^k)
        qc.append(U.power(2**k).control(1), [count[k], *target])
    qc.barrier()

    if inverse_qft:
        iqft = qft_manual(t, barriers=False).inverse().to_gate(label="QFT†")
        qc.append(iqft, count)
    if measure:
        c = ClassicalRegister(t, "c")
        qc.add_register(c)
        qc.measure(count, c)
    return qc

qpe_T = qpe_circuit(TGate(), prep_one, t=3)
qpe_T.draw("mpl", fold=-1)

Before running the whole thing, let us look **inside**, just before the inverse QFT. By the argument above, the counting register should be the Fourier state $|\tilde x\rangle$ with $x = \theta\,2^3 = 1$, the first row of Bloch spheres in Chapter 2.

In [ ]:
inside = qpe_circuit(TGate(), prep_one, t=3, measure=False, inverse_qft=False)
plot_bloch_multivector(Statevector(inside), title="counting qubits 0,1,2 and the target, before QFT†")

Turns of $1/8$, $1/4$ and $1/2$ on counting qubits 0, 1, 2. That is $|\tilde1\rangle$, exactly as on the clock. The target is still $|1\rangle$ at the south pole.

> **Predict.** Which bitstring will the full circuit return, and how often?

In [ ]:
counts = sampler.run([qpe_T], shots=1000).result()[0].data.c.get_counts()
display(plot_histogram(counts, title="QPE on the T gate, t = 3"))

x = int(max(counts, key=counts.get), 2)
print(f"measured x = {x},  estimate theta = x / 2^t = {x}/8 = {x/8}")

Every shot returns `001`, so $\tilde\theta = 1/8$, exactly. Because $1/8$ is a three-bit binary fraction, three counting qubits are enough to get it with certainty.

### The cost of the controlled powers

We cheated slightly. `U.power(2**k)` asks Qiskit for $U^{2^k}$ as a single gate, which is easy for a phase gate because $P(\lambda)^{2^k} = P(2^k\lambda)$. For a general unitary, the only way to apply $U^{2^k}$ is to apply $U$ that many times, so the circuit needs $1 + 2 + \dots + 2^{t-1} = 2^t - 1$ applications of controlled-$U$ in total. Phase estimation is efficient precisely for those unitaries whose high powers can be implemented cheaply. Modular multiplication in Shor's algorithm (Chapter 9) is one, because $a^{2^k} \bmod N$ can be computed classically by repeated squaring.

### Exercise 6: a mystery gate

**Goal.** The gate below has eigenstate $|1\rangle$ and an eigenphase $\theta$ that is a four-bit binary fraction. Use `qpe_circuit` to find $\theta$ and store it as `ex6_theta`. Do not read the angle out of the gate's definition; measure it.

<details>
<summary>Hint</summary>

A four-bit binary fraction needs $t=4$ counting qubits. Measure, convert the bitstring with `int(bits, 2)`, and divide by $2^4$.
</details>

In [ ]:
mystery = PhaseGate(2 * np.pi * 0.6875)     # pretend you cannot see this number
mystery.label = "mystery"

ex6_theta = None

In [ ]:
check("theta of the mystery gate", lambda: np.isclose(ex6_theta, 11/16))

<details>
<summary>Solution</summary>

```python
qc = qpe_circuit(mystery, prep_one, t=4)
counts = sampler.run([qc], shots=1000).result()[0].data.c.get_counts()
bits = max(counts, key=counts.get)            # '1011'
ex6_theta = int(bits, 2) / 2**4               # 11/16 = 0.6875
```

In binary, $\theta = 0.1011$, and the measured bitstring is the binary expansion of $\theta$ read with the most significant digit on the left. Try $t = 3$: the circuit can no longer represent $11/16$ exactly, and the histogram spreads out. That is the subject of the next chapter.
</details>

Further reading: A. Kitaev, [Quantum measurements and the Abelian stabilizer problem](https://arxiv.org/abs/quant-ph/9511026) (1995), and chapter 5.2 of Nielsen and Chuang.

## Chapter 7. Precision: when the phase is not a binary fraction

Most eigenphases are not of the form $x/2^t$. The phase of an energy eigenstate, for example, is essentially never a neat binary fraction. What does QPE do then?

After the controlled powers the counting register is still

$$\frac{1}{\sqrt{2^t}}\sum_{y=0}^{2^t-1} e^{2\pi i\,\theta y}\,|y\rangle ,$$

but now $\theta\,2^t$ is not an integer, so this is not exactly any Fourier basis state. Apply $\mathrm{QFT}^\dagger$ and compute the amplitude of the outcome $x$. Writing $\delta = \theta - x/2^t$ for the distance between $\theta$ and the grid point $x/2^t$, the amplitude is a geometric series:

$$\langle x|\,\mathrm{QFT}^\dagger\cdots = \frac{1}{2^t}\sum_{y=0}^{2^t-1} e^{2\pi i\,\delta y} = \frac{1}{2^t}\,\frac{1 - e^{2\pi i\,2^t\delta}}{1 - e^{2\pi i\,\delta}},$$

and therefore

$$P(x) = \frac{\sin^2(\pi\,2^t\delta)}{4^t\,\sin^2(\pi\delta)} .$$

This is the **Fejér kernel**. It peaks at the grid point closest to $\theta$, with side lobes falling off like $1/(2^t\delta)^2$. Two classic results follow from it:

- The probability of getting the closest grid point is **always at least $4/\pi^2 \approx 0.405$**, however awkward $\theta$ is.
- To get $\theta$ right to $m$ bits with failure probability at most $\varepsilon$, it is enough to use $t = m + \lceil\log_2(2 + 1/(2\varepsilon))\rceil$ counting qubits. The extra qubits push the side lobes out of the way.

> **Physics note.** The function $\sin^2(N\alpha)/\sin^2\alpha$ is the intensity pattern of light behind a grating with $N$ slits. More slits make the main peak narrower and the side maxima weaker, which is why a spectrometer with a large grating resolves finer lines. Here the "slits" are the $2^t$ branches of the counting register. The same function appears in the theory of X-ray diffraction from a finite crystal.

We take $\theta = 1/3$, whose binary expansion $0.010101\ldots$ never ends.

In [ ]:
def qpe_distribution_theory(theta, t):
    x = np.arange(2**t)
    delta = theta - x / 2**t
    with np.errstate(divide="ignore", invalid="ignore"):
        p = np.sin(np.pi * 2**t * delta)**2 / (4**t * np.sin(np.pi * delta)**2)
    p[np.isclose(np.sin(np.pi * delta), 0)] = 1.0          # exact grid point
    return p

theta = 1/3
fig, axes = plt.subplots(1, 3, figsize=(15, 3.6))
for ax, t in zip(axes, [3, 5, 7]):
    U = PhaseGate(2 * np.pi * theta)
    counts = sampler.run([qpe_circuit(U, prep_one, t)], shots=4000).result()[0].data.c.get_counts()
    xs = np.arange(2**t)
    sim = [counts.get(format(x, f"0{t}b"), 0) / 4000 for x in xs]
    ax.bar(xs / 2**t, sim, width=0.7 / 2**t, label="simulated")
    ax.plot(xs / 2**t, qpe_distribution_theory(theta, t), "k.", label="theory")
    ax.axvline(theta, color="C3", ls="--", label=r"true $\theta$")
    ax.set_title(f"t = {t} counting qubits"); ax.set_xlabel(r"estimate $x/2^t$")
    ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

With three counting qubits the estimate is spread over several grid points. With seven, most of the probability sits on the two points next to $1/3$, and the most likely one is within $1/128$ of the truth.

### Interactive: slide the phase between grid points

Move $\theta$ slowly with $t=4$. When $\theta$ lands exactly on a multiple of $1/16$, the histogram is a single bar. Halfway between two grid points it splits into two bars of about $0.405$ each, the worst case.

In [ ]:
@interact(theta=FloatSlider(min=0, max=1, step=1/128, value=0.3, description="theta", readout_format=".4f"))
def slide_phase(theta):
    t = 4
    p = qpe_distribution_theory(theta, t)
    plt.figure(figsize=(8, 3))
    plt.bar(np.arange(2**t) / 2**t, p, width=0.6 / 2**t)
    plt.axvline(theta, color="C3", ls="--")
    plt.ylim(0, 1); plt.xlabel(r"estimate $x/2^t$"); plt.ylabel("probability")
    plt.title(f"t = 4,  P(closest grid point) = {p.max():.3f}")
    plt.show()

### The 4/π² bound, seen directly

In [ ]:
t = 4
fine = np.linspace(0, 1 / 2**t * 2, 400)               # two grid spacings
best = [qpe_distribution_theory(th, t).max() for th in fine]

plt.figure(figsize=(7, 3.5))
plt.plot(fine * 2**t, best)
plt.axhline(4 / np.pi**2, ls="--", c="gray", label=r"$4/\pi^2 \approx 0.405$")
plt.xlabel(r"$\theta \cdot 2^t$  (grid points are integers)")
plt.ylabel("P(closest estimate)")
plt.title("Success probability between grid points, t = 4")
plt.legend(); plt.grid(alpha=.3); plt.show()

### Exercise 7: how many counting qubits?

**Goal.** For $\theta = 1/3$, find the **smallest** $t$ for which the *most likely* outcome $\tilde\theta = x/2^t$ satisfies $|\tilde\theta - 1/3| < 0.01$. Store it as `ex7_t`.

> **Predict.** Each counting qubit halves the grid spacing. A spacing of $1/2^t$ puts the nearest point within $1/2^{t+1}$ of $\theta$. Which $t$ makes that safely smaller than $0.01$? Is the answer sometimes smaller than that rough estimate suggests?

<details>
<summary>Hint</summary>

Loop over $t = 2, 3, \dots$, take `x = qpe_distribution_theory(1/3, t).argmax()`, and print `abs(x / 2**t - 1/3)`. You can confirm with `qpe_circuit` and the sampler.
</details>

In [ ]:
ex7_t = None

In [ ]:
def _ex7():
    err = lambda t: abs(qpe_distribution_theory(1/3, t).argmax() / 2**t - 1/3)
    return err(ex7_t) < 0.01 and err(ex7_t - 1) >= 0.01

check("smallest t that estimates 1/3 to within 0.01", _ex7)

<details>
<summary>Solution</summary>

```python
for t in range(2, 9):
    x = qpe_distribution_theory(1/3, t).argmax()
    print(t, x, x / 2**t, abs(x / 2**t - 1/3))
ex7_t = 6
```

With $t = 5$ the best estimate is $11/32 = 0.34375$, off by $0.0104$, just too much. With $t = 6$ it is $21/64 = 0.328125$, off by $0.0052$.

There is a neat reason. $2^t/3$ is always exactly one third away from the nearest integer, so the closest grid point is always $1/(3\cdot2^t)$ from $\theta$. We need $3\cdot 2^t > 100$, which first happens at $t = 6$. The rough worst-case estimate $1/2^{t+1} < 0.01$ happens to give the same answer here.
</details>

Further reading: section 5.2.1 of Nielsen and Chuang, where the counting-qubit formula above is derived.

## Chapter 8. The library version, energies, and noise

### Qiskit's built-in phase estimation

Qiskit provides the whole construction as a function, `phase_estimation(num_evaluation_qubits, unitary)`. It returns the circuit *without* the eigenstate preparation or the measurements, so we add those. Internally it uses the same controlled powers and an inverse QFT, followed by a qubit permutation.

That permutation means the library uses the **opposite bit order** to ours. It stores the *first* binary digit of $\theta$ on evaluation qubit 0, so its bitstring is the binary expansion of $\theta$ written backwards. We compare the two on a phase whose bits are not a palindrome, $\theta = 11/16 = 0.1011$.

> **Predict.** Our circuit returns `1011`. What will the library return?

In [ ]:
from qiskit.circuit.library import phase_estimation

t = 4
U = PhaseGate(2 * np.pi * 11 / 16)
lib = phase_estimation(t, U)
display(lib.draw("mpl", fold=-1))

full = QuantumCircuit(t + 1, t)
full.x(t)                                    # eigenstate |1> on the last qubit
full.compose(lib, inplace=True)
full.measure(range(t), range(t))

lib_bits  = max(sampler.run([full], shots=1000).result()[0].data.c.get_counts().items(), key=lambda kv: kv[1])[0]
ours_bits = max(sampler.run([qpe_circuit(U, prep_one, t)], shots=1000).result()[0].data.c.get_counts().items(), key=lambda kv: kv[1])[0]

print(f"ours:     {ours_bits}  ->  theta = int(bits, 2) / 2^t       = {int(ours_bits, 2) / 2**t}")
print(f"library:  {lib_bits}  ->  theta = int(bits[::-1], 2) / 2^t = {int(lib_bits[::-1], 2) / 2**t}")

### Estimating an energy

The most important application of QPE outside cryptography is **chemistry and materials**. For a Hamiltonian $H$ with energy eigenstate $|E\rangle$, the time-evolution operator $U = e^{-iH\tau}$ satisfies

$$U|E\rangle = e^{-iE\tau}|E\rangle, \qquad\text{so}\qquad \theta = -\frac{E\tau}{2\pi}\ (\mathrm{mod}\ 1).$$

Measure $\theta$, recover $E$. We take a one-qubit Hamiltonian whose answer we can check by hand,

$$H = \tfrac12\,(X + Z), \qquad E = \pm\tfrac{1}{\sqrt2} \approx \pm 0.7071 .$$

Its ground state is $R_y(-3\pi/4)|0\rangle$ and its excited state is $R_y(\pi/4)|0\rangle$.

On real problems one never builds $e^{-iH\tau}$ as a matrix. It is approximated by a product of simple gates (Trotterisation, provided in Qiskit as `PauliEvolutionGate`). Here the system is tiny, so we compute the exact matrix with `scipy`.

> **Physics note.** Abrams and Lloyd proposed using QPE to compute molecular energies in 1999, and Aspuru-Guzik and co-workers simulated it for small molecules in 2005. QPE is the "fault-tolerant era" method for chemistry: it gives energies to chemical accuracy but needs long circuits. The variational methods you meet later this week (VQE, SQD) were invented to do something useful *before* such circuits are possible.

In [ ]:
from scipy.linalg import expm
from qiskit.quantum_info import SparsePauliOp

H = SparsePauliOp.from_list([("X", 0.5), ("Z", 0.5)])
tau = 1.0
U_H = UnitaryGate(expm(-1j * tau * H.to_matrix()), label="exp(-iHτ)")

def phase_to_energy(theta, tau):
    theta = theta - 1 if theta >= 0.5 else theta        # map [0,1) to [-1/2, 1/2)
    return -2 * np.pi * theta / tau

excited = QuantumCircuit(1); excited.ry(np.pi / 4, 0)
t = 7
counts = sampler.run([qpe_circuit(U_H, excited, t)], shots=2000).result()[0].data.c.get_counts()
x = int(max(counts, key=counts.get), 2)

print("exact eigenvalues:    ", np.round(np.linalg.eigvalsh(H.to_matrix()), 4))
print(f"QPE estimate (t = {t}): E = {phase_to_energy(x / 2**t, tau):.4f}")

The estimate is within the grid spacing $2\pi/(2^t\tau) \approx 0.05$ of $1/\sqrt2$. More counting qubits, or a longer evolution time $\tau$, would sharpen it.

### What if we do not know the eigenstate?

Usually we do not. Suppose we start from some state $|\phi\rangle = \sum_j c_j|E_j\rangle$. QPE is linear, so each eigen-component produces its own peak, and the measurement picks one of them:

$$P(\text{read } E_j) = |c_j|^2 .$$

QPE then behaves as a **projective measurement of the energy**. It returns one eigenvalue at random, and leaves the target in the matching eigenstate. This is the "failure" from Chapter 5 turned into a feature.

> **Predict.** Start from $|0\rangle$. Its overlaps with the two eigenstates are $\cos(\pi/8)$ and $\sin(\pi/8)$. What fraction of shots should give each energy?

In [ ]:
counts = sampler.run([qpe_circuit(U_H, QuantumCircuit(1), t)], shots=4000).result()[0].data.c.get_counts()

positive = sum(c for bits, c in counts.items() if phase_to_energy(int(bits, 2) / 2**t, tau) > 0)
print(f"estimates near E = +0.71:  {positive / 4000:.3f} of shots")
print(f"estimates near E = -0.71:  {1 - positive / 4000:.3f} of shots")
print(f"\nprediction:  cos²(π/8) = {np.cos(np.pi/8)**2:.3f},  sin²(π/8) = {np.sin(np.pi/8)**2:.3f}")

In practice the starting state is chosen to have a large overlap with the ground state, for example from a classical calculation or from VQE. That overlap is the success probability, so preparing a good starting state is one of the central problems of quantum chemistry on quantum computers.

### QPE on a noisy device

Now the bad news. Every counting qubit adds a controlled power and $O(t)$ more two-qubit gates to the inverse QFT. We transpile our T-gate example for the `FakeSherbrooke` snapshot from Essentials Chapter 8 and see how fast the correct answer fades.

In [ ]:
from qiskit_aer import AerSimulator
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

fake = FakeSherbrooke()
noisy_sim = AerSimulator.from_backend(fake)
pm = generate_preset_pass_manager(backend=fake, optimization_level=3, seed_transpiler=11)

ts, success, twoq = [3, 4, 5, 6], [], []
for t in ts:
    isa = pm.run(qpe_circuit(TGate(), prep_one, t))
    ops = isa.count_ops()
    twoq.append(ops.get("ecr", 0) + ops.get("cz", 0))
    counts = noisy_sim.run(isa, shots=4000, seed_simulator=5).result().get_counts()
    correct = format(round(2**t / 8) % 2**t, f"0{t}b")        # exact answer for theta = 1/8
    success.append(counts.get(correct, 0) / 4000)

fig, ax1 = plt.subplots(figsize=(7, 3.5))
ax1.plot(ts, success, "o-", label="P(correct), noisy simulation")
ax1.axhline(1, ls="--", c="gray")
ax1.set_xlabel("counting qubits t"); ax1.set_ylabel("P(correct)"); ax1.set_ylim(0, 1.05)
ax2 = ax1.twinx(); ax2.bar(ts, twoq, alpha=0.25, color="C1"); ax2.set_ylabel("two-qubit gates after transpiling")
ax1.set_title("QPE on a simulated IBM Sherbrooke"); ax1.grid(alpha=.3)
plt.show()
print("P(correct):          ", {t: round(float(p), 3) for t, p in zip(ts, success)})
print("two-qubit gate counts:", dict(zip(ts, twoq)))

In the ideal circuit the right answer comes out every time, for every $t$. On the noisy device the success probability falls steadily as the two-qubit gate count grows. More precision costs more gates, and more gates cost fidelity. This is why full QPE is generally considered an algorithm for **error-corrected** machines.

There are variants designed for today's hardware. **Iterative phase estimation** uses a single counting qubit, measured and reset $t$ times, with classical feed-forward deciding the phase correction for each round. It needs mid-circuit measurement and dynamic circuits, both available on IBM devices.

### Running on IBM Quantum hardware (optional)

The pattern is the same as in Essentials Chapter 9: only the backend changes. The cell is off by default.

In [ ]:
RUN_ON_HARDWARE = False        # set to True once your account is saved (Essentials, Chapter 9)

if RUN_ON_HARDWARE:
    from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as RuntimeSampler

    service = QiskitRuntimeService()
    backend = service.least_busy(operational=True, simulator=False, min_num_qubits=5)

    isa_hw = generate_preset_pass_manager(backend=backend, optimization_level=3).run(qpe_T)
    sampler_hw = RuntimeSampler(mode=backend)
    sampler_hw.options.dynamical_decoupling.enable = True
    sampler_hw.options.twirling.enable_gates = True

    job = sampler_hw.run([isa_hw], shots=2000)
    print("Job ID:", job.job_id())
    hw_counts = job.result()[0].data.c.get_counts()
    display(plot_histogram(hw_counts, title=f"QPE on the T gate, {backend.name}"))

> **Predict.** On hardware, which wrong answers do you expect to be most common for `qpe_T`? Bitstrings one bit-flip away from `001`, or ones close to $1/8$ in value? Why might the answer differ from the ideal Fejér-kernel picture?

Further reading: the [`phase_estimation` entry in the circuit library](https://quantum.cloud.ibm.com/docs/en/api/qiskit/circuit_library), [Build noise models](https://quantum.cloud.ibm.com/docs/en/guides/build-noise-models), and the IBM guide on [classical feedforward and control flow](https://quantum.cloud.ibm.com/docs/en/guides/classical-feedforward-and-control-flow) for iterative phase estimation.

## Chapter 9. A worked problem: factoring 15

We finish with the most famous application of everything above. As in the Essentials notebook, we solve it one decision at a time. Before each code cell there is a question. Think first, then open the answer.

**The problem.** Find a non-trivial factor of $N = 15$.

Obviously the answer is 3 or 5. The point is the method, which is the one that would, on a large enough error-corrected machine, factor a 2048-bit RSA modulus.

### Step 1. Reduce factoring to a period

Pick a number $a$ with $\gcd(a, N) = 1$, say $a = 7$. Consider the sequence $a^0, a^1, a^2, \dots \bmod N$. It must eventually repeat. The smallest $r > 0$ with $a^r \equiv 1 \pmod N$ is the **order** of $a$.

If $r$ is even, then $a^r - 1 = (a^{r/2} - 1)(a^{r/2} + 1) \equiv 0 \pmod N$. So $N$ divides the product, and unless $a^{r/2} \equiv -1$, each factor shares a non-trivial divisor with $N$. We find it with Euclid's $\gcd$, which is fast.

**Question.** What is the order of 7 mod 15, and which factors does it give? Compute it classically first, so we know what the quantum computer should find.

In [ ]:
N, a = 15, 7
print("powers of a mod N:", [pow(a, k, N) for k in range(9)])

r = next(k for k in range(1, N) if pow(a, k, N) == 1)
print("order r =", r)
print("gcd(a^(r/2) - 1, N) =", gcd(pow(a, r // 2) - 1, N))
print("gcd(a^(r/2) + 1, N) =", gcd(pow(a, r // 2) + 1, N))

The sequence is $1, 7, 4, 13, 1, \dots$ with period $r = 4$, and $\gcd(48, 15) = 3$, $\gcd(50, 15) = 5$. Finding $r$ classically took us a loop of length $r$, which for a 2048-bit number is astronomically long. The rest is easy. **Order finding is the hard part, and it is the part we give to the quantum computer.**

### Step 2. Turn order finding into phase estimation

Define the unitary that multiplies by $a$:

$$U_a|y\rangle = |a\,y \bmod N\rangle \quad (y < N), \qquad U_a|y\rangle = |y\rangle \quad (y \ge N).$$

It is a permutation of basis states, so it is unitary (because $\gcd(a,N) = 1$, multiplication by $a$ is invertible mod $N$).

**Question.** We know $U_a^r = I$. What can the eigenvalues of $U_a$ be?

<details>
<summary>Answer</summary>

If $U|u\rangle = \lambda|u\rangle$ then $U^r|u\rangle = \lambda^r|u\rangle = |u\rangle$, so $\lambda^r = 1$. The eigenvalues are $r$-th roots of unity, $e^{2\pi i\,s/r}$ for $s = 0, 1, \dots, r-1$. **The eigenphases are $\theta = s/r$, so phase estimation reveals the order in the denominator.**

The corresponding eigenstates are Fourier states over the orbit $1, a, a^2, \dots$:

$$|u_s\rangle = \frac{1}{\sqrt r}\sum_{k=0}^{r-1} e^{-2\pi i\, sk/r}\,|a^k \bmod N\rangle .$$

The QFT appears again, this time on the orbit of $a$.
</details>

### Step 3. The eigenstate problem

To write $|u_s\rangle$ down we need $r$, which is what we are looking for. That seems circular.

**Question.** What is $\frac{1}{\sqrt r}\sum_s |u_s\rangle$? Look at the formula for $|u_s\rangle$ and sum over $s$ first.

<details>
<summary>Answer</summary>

$\sum_{s} e^{-2\pi i\, sk/r}$ is $r$ when $k = 0$ and $0$ otherwise (the roots of unity cancel, as in the $\mathrm{QFT}|0\rangle$ calculation). So

$$\frac{1}{\sqrt r}\sum_{s=0}^{r-1}|u_s\rangle = |1\rangle .$$

The easy state $|1\rangle$ is an equal superposition of all the eigenstates. By Chapter 8, QPE on $|1\rangle$ returns $s/r$ for a uniformly random $s$. We do not need to know the eigenstates at all.
</details>

Let us check that claim numerically before relying on it.

In [ ]:
n_work = int(np.ceil(np.log2(N)))                # 4 qubits hold the numbers 0..15
orbit = [pow(a, k, N) for k in range(r)]

total = np.zeros(2**n_work, dtype=complex)
for s in range(r):
    u_s = np.zeros(2**n_work, dtype=complex)
    for k, y in enumerate(orbit):
        u_s[y] = np.exp(-2j * np.pi * s * k / r) / np.sqrt(r)
    total += u_s / np.sqrt(r)

print("sum of eigenstates / sqrt(r) equals |1> ?", np.allclose(total, Statevector.from_int(1, 2**n_work).data))

### Step 4. Build the controlled powers

Phase estimation needs controlled-$U_a^{2^k}$. Applying $U_a$ $2^k$ times would cost exponentially many gates.

**Question.** Is there a cheaper way to get $U_a^{2^k}$?

<details>
<summary>Answer</summary>

$U_a^{2^k}$ multiplies by $a$, $2^k$ times, which is the same as multiplying once by $a^{2^k} \bmod N$. That number can be computed **classically** by repeated squaring in $k$ steps. So $U_a^{2^k} = U_{a^{2^k} \bmod N}$, and every controlled power costs the same as a single modular multiplication. This is what makes Shor's algorithm efficient.
</details>

An honest disclaimer: in a real implementation, $U_a$ is a reversible arithmetic circuit built from Toffoli gates, like the adder in Essentials Chapter 10 but much bigger. Here we build it directly as a permutation matrix with `UnitaryGate`. That is fine for learning, but it would be cheating at scale, since writing the matrix down is exponentially expensive.

In [ ]:
def mod_mult_gate(a, N, n_work):
    # Permutation matrix for |y> -> |a*y mod N> (y < N), identity on y >= N.
    dim = 2**n_work
    M = np.zeros((dim, dim))
    for y in range(dim):
        M[(a * y) % N if y < N else y, y] = 1
    return UnitaryGate(M, label=f"×{a} mod {N}")

for k in range(4):
    print(f"k = {k}:  U^(2^k) multiplies by {a}^{2**k} mod {N} = {pow(a, 2**k, N)}")

From $k = 2$ onwards the multiplier is $7^4 \equiv 1$, so those controlled gates do nothing. That is special to this tiny example, where the order happens to be a power of two. In general one uses about $t \approx 2\log_2 N$ counting qubits, so that the continued-fraction step in Step 6 is guaranteed to work even when $r$ does not divide $2^t$.

### Step 5. Assemble and run

Eight counting qubits ($\approx 2\log_2 15$) and four work qubits, initialised to $|1\rangle$. The structure is exactly `qpe_circuit`, with the controlled powers replaced by the cheap ones from Step 4.

> **Predict.** The eigenphases are $s/4$ for $s = 0, 1, 2, 3$, all exact binary fractions. With $t = 8$, which four integers $x$ should the histogram show, and with what probabilities?

In [ ]:
def order_finding_circuit(a, N, t):
    n_work = int(np.ceil(np.log2(N)))
    count = QuantumRegister(t, "count")
    work  = QuantumRegister(n_work, "work")
    c     = ClassicalRegister(t, "c")
    qc = QuantumCircuit(count, work, c)

    qc.x(work[0])                                        # work register = |1>
    qc.h(count)
    for k in range(t):
        a_k = pow(a, 2**k, N)                            # classical repeated squaring
        qc.append(mod_mult_gate(a_k, N, n_work).control(1), [count[k], *work])
    qc.append(qft_manual(t, barriers=False).inverse().to_gate(label="QFT†"), count)
    qc.measure(count, c)
    return qc

t = 8
shor15 = order_finding_circuit(a=7, N=15, t=t)
display(shor15.draw("mpl", fold=-1, scale=0.7))

counts = sampler.run([shor15], shots=2000).result()[0].data.c.get_counts()
plot_histogram(counts, title="Order finding for a = 7, N = 15")

Four peaks at $x = 0, 64, 128, 192$, each about a quarter of the shots. Dividing by $2^8 = 256$ gives $\tilde\theta = 0, 1/4, 1/2, 3/4$, which is $s/r$ for $s = 0, 1, 2, 3$ and $r = 4$. Compare with Chapter 4: this is a periodic spectrum with peaks at multiples of $N/r = 256/4$.

### Step 6. Post-process: from a phase to the order

We measured $\tilde\theta \approx s/r$, but we see only the fraction, not $r$ itself.

**Question.** Given $\tilde\theta = 0.75$, what is the most natural guess for $r$? And what goes wrong for $\tilde\theta = 0.5$?

<details>
<summary>Answer</summary>

$0.75 = 3/4$, so $r = 4$. But $0.5 = 2/4 = 1/2$ reduces to denominator 2, which is a *divisor* of $r$, not $r$ itself. And $\tilde\theta = 0$ tells us nothing. We therefore check each candidate classically ($a^r \equiv 1$?) and simply repeat the experiment when it fails.

In general $\tilde\theta$ is only *close* to $s/r$, and the right tool for finding the nearby fraction with a small denominator is the **continued fraction expansion**. In Python, `Fraction(x, 2**t).limit_denominator(N)` does it, since we know $r < N$.
</details>

In [ ]:
print(" measured x | theta  | fraction | candidate r | a^r mod N = 1? | factors")
print("------------+--------+----------+-------------+----------------+---------")
found = set()
for bits in sorted(counts, key=lambda b: int(b, 2)):
    x = int(bits, 2)
    frac = Fraction(x, 2**t).limit_denominator(N)
    r_guess = frac.denominator
    ok = pow(a, r_guess, N) == 1 and r_guess % 2 == 0
    factors = ""
    if ok:
        f1, f2 = gcd(pow(a, r_guess // 2) - 1, N), gcd(pow(a, r_guess // 2) + 1, N)
        factors = f"{f1} × {f2}"
        found |= {f1, f2} - {1, N}
    print(f"   {x:5d}    | {x/2**t:.4f} | {str(frac):>8s} | {r_guess:11d} | {str(ok):>14s} | {factors}")

print("\nnon-trivial factors found:", sorted(found))

**What we learned.**

- Half the outcomes ($s = 1$ and $s = 3$) give $r = 4$ immediately. The other half fail and we would just run again. The expected number of repetitions is small, and it stays small for large $N$.
- Every piece of this notebook appeared: the QFT turned a periodic phase pattern into sharp peaks (Chapter 4), kickback wrote the eigenphases into the counting register (Chapter 5), phase estimation read them out (Chapter 6), and the continued fractions handle the case where the phase is not an exact binary fraction (Chapter 7).
- The only quantum step is order finding. Everything else, choosing $a$, repeated squaring, continued fractions and $\gcd$, is classical and fast.

### Exercise 8: factoring 21

**Goal.** Factor $N = 21$ with $a = 2$. Use `order_finding_circuit` with enough counting qubits ($t = 10$ works well), run it, and post-process the counts with continued fractions as above. Store the order you find as `ex8_order` and the set of non-trivial factors as `ex8_factors`.

> **Predict.** Compute $2^k \bmod 21$ by hand for a few $k$. What is the order? Will the peaks sit exactly on integers this time?

<details>
<summary>Hint</summary>

You need $n_\text{work} = 5$ work qubits for numbers up to 20, which `order_finding_circuit` handles automatically. The order is 6, which does not divide $2^{10}$, so the peaks are spread out exactly as in Chapter 7. `limit_denominator(21)` still recovers the fraction. Expect some outcomes to give $r = 2$ or $r = 3$, divisors of the true order, and discard them with the check $a^r \bmod N = 1$.
</details>

In [ ]:
ex8_order   = None
ex8_factors = None      # a Python set, e.g. {x, y}

In [ ]:
check("order of 2 mod 21", lambda: ex8_order == 6)
check("factors of 21",      lambda: set(ex8_factors) == {3, 7})

<details>
<summary>Solution</summary>

```python
N, a, t = 21, 2, 10
counts = sampler.run([order_finding_circuit(a, N, t)], shots=2000).result()[0].data.c.get_counts()

orders, factors = set(), set()
for bits in counts:
    r_guess = Fraction(int(bits, 2), 2**t).limit_denominator(N).denominator
    if pow(a, r_guess, N) == 1 and r_guess % 2 == 0:
        orders.add(r_guess)
        factors |= {gcd(pow(a, r_guess // 2) - 1, N), gcd(pow(a, r_guess // 2) + 1, N)} - {1, N}

ex8_order = min(orders)          # 6
ex8_factors = factors            # {3, 7}
```

You may also see a candidate $r = 12$. It passes the check because any multiple of the order does, so we take the smallest.

$2^3 = 8$, and $\gcd(7, 21) = 7$, $\gcd(9, 21) = 3$. Look at the histogram: the peaks sit near multiples of $1024/6 \approx 170.7$, each one a small Fejér kernel. This circuit has 15 qubits, close to the limit of what is pleasant to simulate exactly in a notebook.
</details>

**Stretch.** Try $a = 4$ with $N = 15$. What order do you get, and why does it still factor 15? Then try $a = 14$. Why does it fail? (Hint: $14 \equiv -1 \pmod{15}$.)

Further reading: P. Shor, [Polynomial-time algorithms for prime factorization and discrete logarithms on a quantum computer](https://arxiv.org/abs/quant-ph/9508027) (1995), and the factoring lesson in [Fundamentals of quantum algorithms](https://quantum.cloud.ibm.com/learning/en/courses/fundamentals-of-quantum-algorithms).

## Where we go next

QFT and phase estimation are subroutines inside many larger algorithms.

| Topic | How it uses this notebook |
|---|---|
| Shor's algorithm (full scale) | Order finding (Chapter 9) with modular arithmetic built from reversible adders |
| Quantum chemistry with QPE | Energy estimation with $U = e^{-iH\tau}$ (Chapter 8), Trotterised with `PauliEvolutionGate` |
| Amplitude estimation | QPE applied to the Grover operator; a quadratic speed-up for Monte Carlo estimates |
| HHL (linear systems) | QPE to write eigenvalues into a register, then a controlled rotation by $1/\lambda$ |
| Iterative and Bayesian phase estimation | One counting qubit plus mid-circuit measurement; suited to near-term devices |
| Quantum error correction | Syndrome extraction is a one-bit phase estimation of a stabiliser whose eigenvalues are $\pm1$ (Chapter 5, the Hadamard test) |
| Variational methods (VQE, SQD) | What we use *instead* of QPE while circuits must stay short (Chapter 8) |

## Quick reference

```python
QFTGate(n)                                   # the n-qubit QFT as one gate
qft_manual(n)                                # our version: H + controlled phases + swaps
qc.cp(lam, m, j)                             # controlled phase; symmetric in m and j
circuit.inverse()                            # reverse the gates, conjugate the angles (QFT -> QFT†)
Statevector(v).evolve(QFTGate(n))            # apply the QFT to an arbitrary state

gate.control(1)                              # controlled version of any gate
gate.power(2**k)                             # U^(2^k) as a single gate
phase_estimation(t, U)                       # library QPE (no prep, no measurement; read bits[::-1])
UnitaryGate(matrix)                          # any unitary from a matrix (small systems only)

theta_estimate = int(bits, 2) / 2**t         # read a QPE result
Fraction(x, 2**t).limit_denominator(N)       # continued fractions: phase -> s/r
average_gate_fidelity(Operator(A), Operator(B))   # how close two circuits are
```

**Bit ordering, once more.** In our `qpe_circuit`, counting qubit $k$ controls $U^{2^k}$ and is bit $k$ of the measured integer. Qiskit prints qubit 0 on the right, so the bitstring reads as the binary expansion of $\tilde\theta$ with the most significant digit on the left. The library's `phase_estimation` uses the reverse order, so reverse its bitstring first.

## Further reading

IBM Quantum documentation:
- [Circuit library](https://quantum.cloud.ibm.com/docs/en/api/qiskit/circuit_library) (`QFTGate`, `phase_estimation`, `PauliEvolutionGate`)
- [Synthesis module](https://quantum.cloud.ibm.com/docs/en/api/qiskit/synthesis) (`synth_qft_full` with approximation)
- [Classical feedforward and control flow](https://quantum.cloud.ibm.com/docs/en/guides/classical-feedforward-and-control-flow), for iterative phase estimation
- [Build noise models](https://quantum.cloud.ibm.com/docs/en/guides/build-noise-models)

Free courses on IBM Quantum Learning:
- [Fundamentals of quantum algorithms](https://quantum.cloud.ibm.com/learning/en/courses/fundamentals-of-quantum-algorithms), which has full lessons on phase estimation and factoring

Original papers:
- A. Kitaev, [Quantum measurements and the Abelian stabilizer problem](https://arxiv.org/abs/quant-ph/9511026) (1995)
- P. Shor, [Polynomial-time algorithms for prime factorization and discrete logarithms](https://arxiv.org/abs/quant-ph/9508027) (1995)
- R. Cleve, A. Ekert, C. Macchiavello, M. Mosca, [Quantum algorithms revisited](https://arxiv.org/abs/quant-ph/9708016) (1998)
- D. Coppersmith, [An approximate Fourier transform useful in quantum factoring](https://arxiv.org/abs/quant-ph/0201067) (1994)

Textbook for the mathematics: Nielsen and Chuang, *Quantum Computation and Quantum Information*, chapter 5.